# Adaptive NeSy-Gen: Claim-Level Verification and Explanations

This notebook reuses an existing Vision-T5 checkpoint. It adds leakage-free visual RAG, uncertainty-triggered PrimeKG/LTN verification, the Consistency Gate, evidence-bound selective revision, and faithful claim-level traces. No training is performed here.

## Cell 1: Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Configuration

In [ ]:
from pathlib import Path

RUN_DATASET = 'iuxray_official'  # options: iuxray_official, mimic_aug
SOURCE_RUN_NAME = 'aaai_vision_t5_base_convnext_primekg'
ADAPTIVE_RUN_NAME = 'aaai_adaptive_nesygen_claim_verification'
SMOKE_RUN = False
MAX_TEST_EXAMPLES = 50 if SMOKE_RUN else None
ABLATION_SMOKE_RUN = True  # Fast Cell 11 check; set False for final paper ablations.
ABLATION_LIMIT = 100 if ABLATION_SMOKE_RUN else None

REPO_DIR = Path('/content/Nesy-GenRevision')
AAAI_ROOT = Path('/content/drive/MyDrive/aaai_2026_experiments')
SOURCE_OUTPUT_DIR = AAAI_ROOT / RUN_DATASET / SOURCE_RUN_NAME
OUTPUT_DIR = AAAI_ROOT / RUN_DATASET / ADAPTIVE_RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_NAME = 'iuxray_official_manifest.jsonl' if RUN_DATASET == 'iuxray_official' else 'mimic_aug_manifest.jsonl'
MANIFEST = SOURCE_OUTPUT_DIR / MANIFEST_NAME
RAD_PRIMEKG_DIR = Path(f'/content/drive/MyDrive/primekg_radiology_cache_{RUN_DATASET}')
VISION_T5_CKPT = SOURCE_OUTPUT_DIR / 'checkpoints' / 'vision_t5_convnext_base_flan-t5-base'

DECODING_MODE = 'graph_constrained'
RETRIEVAL_TOP_K = 10
GENERATOR_NUM_CANDIDATES = 8
GENERATOR_NUM_BEAMS = 8
GENERATOR_BATCH_SIZE = 2
MAX_NEW_TOKENS = 140
GENERATOR_NUM_BEAM_GROUPS = 1
GENERATOR_DIVERSITY_PENALTY = 0.0

# Proposed adaptive routing profile.
FAST_ACCEPT_THRESHOLD = 0.85
MIN_SUPPORTING_REPORTS = 2
CLAIM_REVISE_THRESHOLD = 0.50
REVISION_POLICY = 'evidence_replace'  # options: evidence_replace, audit_only

print('Checkpoint:', VISION_T5_CKPT)
print('Manifest:', MANIFEST)
print('PrimeKG:', RAD_PRIMEKG_DIR)
print('Adaptive output:', OUTPUT_DIR)

## Cell 3: Clone/Pull and Install

In [ ]:
import subprocess, sys

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/FaezehMillerAI/Nesy-GenRevision.git', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[torch,eval,colab]'], check=True)
print('Repository ready:', REPO_DIR)

## Cell 4: Verify Existing Assets (No Retraining)

In [ ]:
import torch

required = [MANIFEST, VISION_T5_CKPT, RAD_PRIMEKG_DIR / 'kg.csv', RAD_PRIMEKG_DIR / 'nodes.csv']
for path in required:
    print(path, '->', path.exists())
assert all(path.exists() for path in required), 'Fix the missing Drive path(s) in Cell 2.'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Checkpoint files:', [path.name for path in list(VISION_T5_CKPT.iterdir())[:12]])

## Cell 5: Run Adaptive RAG + PrimeKG + LTN + Gate

The progress stream has separate bars for the visual index, query images, candidate generation, and adaptive claim verification.

In [ ]:
import os, subprocess, sys

PREDICTIONS_CSV = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_predictions.csv'
CANDIDATES_CSV = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_candidate_audit.csv'
CLAIM_TRACE_JSONL = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_claim_traces.jsonl'
CLAIM_AUDIT_CSV = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_claim_audit.csv'
GENERATION_LOG = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_generation.log'

cmd = [
    sys.executable, '-u', 'scripts/generate_rag_primekg_reports.py',
    '--manifest', str(MANIFEST), '--primekg-dir', str(RAD_PRIMEKG_DIR),
    '--output-csv', str(PREDICTIONS_CSV), '--candidates-csv', str(CANDIDATES_CSV),
    '--claim-trace-jsonl', str(CLAIM_TRACE_JSONL), '--claim-audit-csv', str(CLAIM_AUDIT_CSV),
    '--split', 'test', '--retrieval-mode', 'visual', '--retrieval-top-k', str(RETRIEVAL_TOP_K),
    '--generator-checkpoint-dir', str(VISION_T5_CKPT),
    '--generator-num-candidates', str(GENERATOR_NUM_CANDIDATES),
    '--generator-num-beams', str(GENERATOR_NUM_BEAMS),
    '--generator-batch-size', str(GENERATOR_BATCH_SIZE),
    '--generator-repetition-penalty', '1.12', '--generator-no-repeat-ngram-size', '3',
    '--generator-length-penalty', '0.90',
    '--generator-num-beam-groups', str(GENERATOR_NUM_BEAM_GROUPS),
    '--generator-diversity-penalty', str(GENERATOR_DIVERSITY_PENALTY),
    '--max-new-tokens', str(MAX_NEW_TOKENS), '--decoding-mode', DECODING_MODE,
    '--graph-token-boost', '1.5', '--constraint-max-terms', '2500',
    '--subgraph-strategy', 'ego', '--verification-mode', 'adaptive_claim',
    '--fast-accept-threshold', str(FAST_ACCEPT_THRESHOLD),
    '--min-supporting-reports', str(MIN_SUPPORTING_REPORTS),
    '--claim-revise-threshold', str(CLAIM_REVISE_THRESHOLD),
    '--revision-policy', REVISION_POLICY,
]
if MAX_TEST_EXAMPLES:
    cmd += ['--limit', str(MAX_TEST_EXAMPLES)]

env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
with GENERATION_LOG.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=0, env=env)
    while True:
        chunk = process.stdout.read(1)
        if chunk == '' and process.poll() is not None: break
        if chunk:
            print(chunk, end='', flush=True); log_file.write(chunk); log_file.flush()
    return_code = process.wait()
if return_code != 0: raise RuntimeError(f'Generation failed. See {GENERATION_LOG}')
print('Predictions:', PREDICTIONS_CSV)
print('Faithful traces:', CLAIM_TRACE_JSONL)

## Cell 6: Evaluate Explainability and Routing Efficiency

In [ ]:
import json, subprocess, sys

EXPLAIN_JSON = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_explainability.json'
EXPLAIN_CSV = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_explainability_claims.csv'
subprocess.run([sys.executable, 'scripts/evaluate_adaptive_explanations.py',
    '--trace-jsonl', str(CLAIM_TRACE_JSONL), '--output-json', str(EXPLAIN_JSON),
    '--output-csv', str(EXPLAIN_CSV)], cwd=REPO_DIR, check=True)
json.loads(EXPLAIN_JSON.read_text())

## Cell 7: Inspect Claim-Level Decisions

In [ ]:
import pandas as pd
from IPython.display import display

claims = pd.read_csv(EXPLAIN_CSV)
display(claims[['study_id', 'original_claim', 'entity_names', 'retrieval_support',
                'ltn_truth', 'primekg_status', 'decision', 'reason', 'final_claim']].head(30))
display(claims['decision'].value_counts().rename_axis('decision').to_frame('claims'))

## Cell 8: Build Readable HTML Evidence Report

In [ ]:
import subprocess, sys
from IPython.display import HTML, display

EXPLANATION_HTML = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_explanations.html'
subprocess.run([sys.executable, 'scripts/build_adaptive_explanation_report.py',
    '--trace-jsonl', str(CLAIM_TRACE_JSONL), '--output-html', str(EXPLANATION_HTML),
    '--limit', '20'], cwd=REPO_DIR, check=True)
display(HTML(EXPLANATION_HTML.read_text()))

## Cell 9: Generation and Leakage Evaluation

In [ ]:
import subprocess, sys, json

METRICS_JSON = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_generation_metrics.json'
LEAKAGE_JSON = OUTPUT_DIR / f'{RUN_DATASET}_adaptive_leakage_audit.json'
subprocess.run([sys.executable, 'scripts/evaluate_generation.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(PREDICTIONS_CSV), '--output-json', str(METRICS_JSON),
    '--nodes-csv', str(RAD_PRIMEKG_DIR / 'nodes.csv'),
    '--output-factuality-csv', str(OUTPUT_DIR / f'{RUN_DATASET}_adaptive_entity_factuality.csv'),
    '--output-chexpert-csv', str(OUTPUT_DIR / f'{RUN_DATASET}_adaptive_chexpert_quick.csv'),
    '--output-radgraph-csv', str(OUTPUT_DIR / f'{RUN_DATASET}_adaptive_radgraph_quick.csv')], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'scripts/audit_prediction_leakage.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(PREDICTIONS_CSV), '--output-json', str(LEAKAGE_JSON),
    '--high-overlap-threshold', '0.95'], cwd=REPO_DIR, check=True)
print(json.dumps(json.loads(METRICS_JSON.read_text())['lexical_metrics'], indent=2))
print(LEAKAGE_JSON.read_text())

## Cell 10: Prepare Official CheXbert and RadGraph Inputs

In [ ]:
import subprocess, sys

OFFICIAL_INPUT_DIR = OUTPUT_DIR / 'official_metric_inputs'
subprocess.run([sys.executable, 'scripts/evaluate_generation.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(PREDICTIONS_CSV),
    '--output-json', str(OUTPUT_DIR / f'{RUN_DATASET}_official_input_prep.json'),
    '--prepare-official-inputs-dir', str(OFFICIAL_INPUT_DIR),
    '--nodes-csv', str(RAD_PRIMEKG_DIR / 'nodes.csv')], cwd=REPO_DIR, check=True)
print('Official metric inputs:', OFFICIAL_INPUT_DIR)

## Cell 11: Optional Paper Ablation Suite

Run after the proposed method succeeds. It compares retrieval, raw generator, report-level graph verification, adaptive audit-only, adaptive revision, and always-on claim verification. Candidate generation is cached across compatible variants, and `--resume` skips completed artifacts. Keep `ABLATION_SMOKE_RUN=True` for a fast 100-case check; set it to `False` only for the final paper run.

In [ ]:
import subprocess, sys

ABLATION_PROFILE = f'smoke_n{ABLATION_LIMIT}' if ABLATION_LIMIT else 'full'
ABLATION_DIR = OUTPUT_DIR / 'ablations' / ABLATION_PROFILE
print('Ablation profile:', ABLATION_PROFILE, '| output:', ABLATION_DIR)
cmd = [sys.executable, '-u', 'scripts/run_ablation_suite.py', '--manifest', str(MANIFEST),
    '--primekg-dir', str(RAD_PRIMEKG_DIR), '--output-dir', str(ABLATION_DIR),
    '--dataset-name', RUN_DATASET, '--generator-checkpoint-dir', str(VISION_T5_CKPT),
    '--retrieval-mode', 'visual', '--batch-size', str(GENERATOR_BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS), '--resume']
if ABLATION_LIMIT: cmd += ['--limit', str(ABLATION_LIMIT)]
subprocess.run(cmd, cwd=REPO_DIR, check=True)